In [ ]:
import pandas as pd
import os

folder = os.path.join('..', 'ASEC_RAW')
file = 'asec2019.dat'
input_file = os.path.join(folder, file)
output_file = 'ASEC_NJ_2019.csv'

if not os.path.exists(input_file):

    print(f"ERROR")

else:
    print(f"{file}")

# Household Variables
h_specs = [
    (0, 1),      # House Bracket
    (1, 6),      # Merge key
    (43, 45),    # State (GESTFIPS)
    (43,48),     # County (GTCBSA) (2005 and forward)
    (50, 51),    # Sub/Urb (HCCC-R) (GTCBSAST)
    (81, 83),    # Household Size (H-NUMPER) (HRNUMHOU)
    (88, 89),    # Tenure (H-TENURE)
    (105, 113),  # Total Income (HTOTVAL)
    (33, 41),  # Supplemental Weight (HSUP_WGT)
]
h_names = ['Housing_Variable', 'Merge', 'State','County', 'Sub/Urb', 'HH_Size', 'Tenure', 'HH_Income', 'Supp_Weight']

# Person Variables
p_specs = [
    (0, 1),      # Person Bracket
    (1, 6),      # Merge key
    (81, 83),    # Relationship (A-EXPRRP)
    (54, 62),    # Weight (A-FNLWGT)
    (78, 80),    # Age (A-AGE) (PEAGE)
    (86, 88),    # Education (A-HGA) (PEEDUCA)
    (89, 90),    # Marital Status (A-MARITL) (PRMARSTA)
    (577, 585),  # Person Income (PTOTVAL) 
]
p_names = ['Person_Variable', 'Merge', 'Relationship', 'Weight', 'Age', 'Education', 'Marital', 'Person_Income']

households = []
persons = []
nj_house = False

with open(input_file, 'r') as f:
    for line in f:
        
        line = line.replace('\n', '').ljust(500) 
        rectype = line[0]

        if rectype == '1': 
            # Filter for NJ '34'
            if line[41:43] == '34':

                nj_house = True  
                data = [line[s:e].strip() for s, e in h_specs]
                households.append(data)

            else:

                nj_house = False 

        elif rectype == '3':

            if nj_house and line[81:83] in ['01', '02']:

                data = [line[s:e].strip() for s, e in p_specs]
                persons.append(data)

df_h = pd.DataFrame(households, columns=h_names)
df_p = pd.DataFrame(persons, columns=p_names)

nj_df = pd.merge(df_h, df_p, on='Merge', how='inner')

numeric_columns = ['HH_Size', 'Age', 'HH_Income', 'Person_Income']

for col in numeric_columns:

    nj_df[col] = pd.to_numeric(nj_df[col], errors='coerce') 

nj_df['Weight'] = pd.to_numeric(nj_df['Weight'], errors='coerce') / 100

nj_df['Year'] = 2019

nj_df = nj_df.drop(columns=['Housing_Variable', 'Person_Variable'])

nj_df.to_csv(output_file, index=False)


asec2019.dat


In [ ]:
import pandas as pd
import os

folder = os.path.join('..', 'ASEC_RAW')
h_file = os.path.join(folder, 'hhpub23.csv')
p_file = os.path.join(folder, 'pppub23.csv')
output_file = 'ASEC_NJ_2023.csv'


h_cols = [
    'H_SEQ',  
    'GESTFIPS',    
    'GTCBSA',    
    'GTCBSAST',   
    'H_NUMPER',    
    'H_TENURE',    
    'HTOTVAL',   
    'HSUP_WGT'     
]

p_cols = [
    'PH_SEQ',     
    'A_EXPRRP',   
    'A_FNLWGT',   
    'A_AGE',      
    'A_HGA',     
    'A_MARITL',   
    'PTOTVAL'   
]

df_h = pd.read_csv(h_file, usecols=h_cols)

df_h = df_h[df_h['GESTFIPS'] == 34]

df_p = pd.read_csv(p_file, usecols=p_cols)

df_p = df_p[df_p['A_EXPRRP'].isin([1, 2])]

nj_df = pd.merge(df_h, df_p, left_on='H_SEQ', right_on='PH_SEQ', how='inner')

nj_df['A_FNLWGT'] = nj_df['A_FNLWGT'] / 100

nj_df['Year'] = 2023

nj_df = nj_df.rename(columns={
    'GESTFIPS': 'State',
    'GTCBSA': 'County',      
    'GTCBSAST': 'Sub/Urb',
    'H_NUMPER': 'HH_Size',
    'H_TENURE': 'Tenure',
    'HTOTVAL': 'HH_Income',
    'HSUP_WGT': 'Supp_Weight',
    'A_EXPRRP': 'Relationship',
    'A_FNLWGT': 'Weight',
    'A_AGE': 'Age',
    'A_HGA': 'Education',
    'A_MARITL': 'Marital',
    'PTOTVAL': 'Person_Income'
})

nj_df = nj_df.drop(columns=['H_SEQ', 'PH_SEQ'])

column_order = [
    'State', 'County', 'Sub/Urb', 'HH_Size', 'Tenure', 'HH_Income', 
    'Supp_Weight', 'Relationship', 'Weight', 'Age', 'Education', 
    'Marital', 'Person_Income', 'Year'
]

nj_df = nj_df[column_order]

nj_df.to_csv(output_file, index=False)


In [13]:
import pandas as pd
import glob

csv_files = glob.glob("*.csv")

df_list = [pd.read_csv(file) for file in csv_files]
merged_df = pd.concat(df_list, ignore_index=True)

merged_df.to_csv("CPS_2005_2023.csv", index=False)


In [14]:
import pandas as pd

input_file = 'CPS_2005_2023.csv'
output_file = 'CLEANED_CPS_2005_2023.csv'

df = pd.read_csv(input_file)

if 'Merge' in df.columns:
    df = df.drop(columns=['Merge'])

column_order = [
    'State', 'County', 'Sub/Urb', 'HH_Size', 'Tenure', 'HH_Income', 
    'Supp_Weight', 'Relationship', 'Weight', 'Age', 'Education', 
    'Marital', 'Person_Income', 'Year'
]

df = df[column_order]

df.to_csv(output_file, index=False)
